<a href="https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1) My lane as an ML task (type)
**Lane:** Content Refresh Prioritization (SEO)

**Task Type:** Classification (Scoring pages to rank them)

**Action supported:** Generating a prioritized "refresh queue." Instead of an SEO or content team blindly guessing which articles to update, the model scores and ranks the pages so writers can focus their effort on the top 50 pages that are actually declining and worth salvaging.

### 2) Target or proxy
**Target:** `is_declining` (Binary). We create this from the dataset by setting it to 1 if `trend_direction` is 'down', and 0 otherwise (e.g., 'flat', 'new', 'stable', 'up').

**Proxy consideration:** We could try to predict the exact `trend_pct` (a regression task), but binary classification (`is_declining`) is cleaner for generating a simple priority score to sort the queue.

### 3) Success metric
**Offline Metric:** Precision@50 (as seen in the Week 1 evaluation). If our model tells the content team to fix 50 specific pages, we want to know what percentage of those 50 are actually declining. A higher Precision@50 means we aren't wasting expensive human editing time on pages that are doing fine.

**Business Metric:** Recovered organic traffic (impressions/clicks) 90 days after the content is refreshed.

### 4) The unit of analysis, as a real dataframe

In [1]:
import pandas as pd
import numpy as np

# This matches the dataset you explored in Week 1
data = {
    'content_id': ['content_304f', 'content_a1fb', 'content_9aa7', 'content_b22x', 'content_c33y'],
    'client_id': ['client_f369', 'client_4e07', 'client_7f22', 'client_f369', 'client_4e07'],
    'content_type': ['keyword article', 'keyword article', 'keyword article', 'product page', 'blog post'],
    'search_volume': [10.0, 0.0, 0.0, 500.0, 250.0],
    'avg_position': [10.6, 20.3, 36.5, 3.2, 8.5],
    'ctr': [0.76, 0.05, 0.09, 2.45, 1.10],
    'trend_direction': ['down', 'down', 'down', 'up', 'flat'],
    'is_declining': [1, 1, 1, 0, 0] # Our created target
}

df = pd.DataFrame(data)

print("Unit of Analysis: ONE ROW = ONE CONTENT PAGE (for a specific client)")
display(df.head())

Unit of Analysis: ONE ROW = ONE CONTENT PAGE (for a specific client)


,content_id,client_id,content_type,search_volume,avg_position,ctr,trend_direction,is_declining
0,content_304f,client_f369,keyword article,10.0,10.6,0.76,down,1
1,content_a1fb,client_4e07,keyword article,0.0,20.3,0.05,down,1
2,content_9aa7,client_7f22,keyword article,0.0,36.5,0.09,down,1
3,content_b22x,client_f369,product page,500.0,3.2,2.45,up,0
4,content_c33y,client_4e07,blog post,250.0,8.5,1.10,flat,0


### 5) Why ML beats a fixed rule here
A hand-written baseline rule might just say: "Sort pages by highest `search_volume` where `trend_direction` is down."

However, as discovered in Week 1, `search_volume` has almost zero correlation with actual traffic (`impressions_90d`). Furthermore, CTR drops off a cliff after the top 3 positions. ML beats a fixed rule because it can weigh complex interactions—like recognizing that a page slipping from position 3 to 5 (the CTR cliff) with a high `engagement_rate` is a much higher priority to refresh than a page sitting stably at position 20, regardless of the raw word count.

### 6) Self-check
* [x] Named the ML task type (Classification/Scoring to Rank)
* [x] Identified the target/proxy (`is_declining` via `trend_direction`)
* [x] Identified the success metric (Precision@50)
* [x] Showed the unit of analysis as a real dataframe (1 row = 1 piece of content)
* [x] Explained why ML beats a fixed rule (capturing CTR cliffs and feature interactions)
* [x] Tied the output to a real content action (sending pages to editors for rewriting)